In [1]:
from pathlib import Path
import sys
import pandas as pd

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src").is_dir()
)
sys.path.insert(0, str(project_root))

from src.data.loader import DatasetLoader
from src.utils.config import Config

# DATASET = "CIC-IoT2023"
DATASET = "MITS-Network"
loader = DatasetLoader(Config())
df = loader.load(DATASET, sample_size=1000)

print(f"Dataset: {DATASET}")
print(f"Shape: {df.shape}")

Dataset: MITS-Network
Shape: (1000, 26)


In [2]:
from pathlib import Path
import sys

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src").is_dir()
)
sys.path.insert(0, str(project_root))

from src.evaluation.dss import DatasetDSSAnalyzer

analyzer = DatasetDSSAnalyzer(
    df=df,
    dataset_name=DATASET
)

dss = analyzer.calculate()

print(f"DSS for {DATASET}: {dss:.3f} / 5.000")
print(f"DSS percentage: {(dss / 5) * 100:.2f}%")

DSS for MITS-Network: 2.650 / 5.000
DSS percentage: 53.00%


In [3]:
result = analyzer.detailed_result()

pd.DataFrame([result])

,Dataset,Alert/Event,Context,Threat Intelligence,Exploitability,Historical,Ground Truth,Scalability,DSS,DSS (%),Analyzed Rows,Total Rows
0,MITS-Network,4.0,4.5,1.0,1.0,4.0,0.0,1.0,2.65,53.0,1000,None


In [4]:
evidence_df = analyzer.evidence_table()

display(evidence_df)

,Dataset,Criterion,Score,Weight,Weighted Score,Evidence
0,MITS-Network,Alert/Event,4.0,0.20,0.80,Alert/event fields: ['event_type'] | Temporal ...
1,MITS-Network,Context,4.5,0.20,0.90,"Native context fields: ['src_hostname', 'dst_h..."
2,MITS-Network,Threat Intelligence,1.0,0.15,0.15,External TI enrichment potential via network i...
3,MITS-Network,Exploitability,1.0,0.10,0.10,Asset mapping permits external CVE/CVSS/exploi...
4,MITS-Network,Historical,4.0,0.15,0.60,"Temporal fields: ['timestamp', 'duration_ms'] ..."
5,MITS-Network,Ground Truth,0.0,0.10,0.00,No ground-truth/label field detected
6,MITS-Network,Scalability,1.0,0.10,0.10,Full dataset size unavailable; analyzed sample...


In [5]:
# Discover non-empty dataset directories using their actual folder names.
DATASETS = sorted(
    path.name
    for path in Config.DATA_ROOT.iterdir()
    if path.is_dir()
    and any(
        file.is_file() and file.suffix.lower() in {".csv", ".parquet"}
        for file in path.rglob("*")
    )
)

print(f"Available datasets: {DATASETS}")

Available datasets: ['BCCC-CIC-IDS-2017', 'BCCC-CSE-IDS2018', 'BigFlow-Parquet', 'CIC-IoT2023', 'MITS-Network', 'TON-IoT', 'UNSW-NB15']


In [ ]:
all_results = []
all_evidence = []

for dataset_name in DATASETS:

    print("=" * 70)
    print(f"Processing: {dataset_name}")

    try:
        # Development/testing sample: keeps memory use manageable.
        df = loader.load(dataset_name, sample_size=10000)

        # IMPORTANT: obtain full-dataset metadata separately so that the
        # scalability criterion is not distorted by the 10k-row sample.
        metadata = loader.metadata(dataset_name)

        print(f"Shape analyzed: {df.shape}")
        print(f"Full dataset rows: {metadata['total_rows']:,}"
              if metadata["total_rows"] is not None
              else "Full dataset rows: unavailable")

        analyzer = DatasetDSSAnalyzer(
            df=df,
            dataset_name=dataset_name,
            total_rows=metadata["total_rows"],
            source_files=metadata["files"],
            sample_size=10000,
        )

        result = analyzer.detailed_result()
        all_results.append(result)

        evidence = analyzer.evidence_table()
        all_evidence.append(evidence)

        print(
            f"DSS = {result['DSS']:.3f}/5 "
            f"({result['DSS (%)']:.2f}%)"
        )

    except Exception as error:
        print(f"ERROR processing {dataset_name}: {error}")


Processing: BCCC-CIC-IDS-2017


In [ ]:
dss_table = pd.DataFrame(all_results)

evidence_table = pd.concat(
    all_evidence,
    ignore_index=True
)

In [ ]:
dss_ranking = (
    dss_table
    .sort_values(by="DSS", ascending=False)
    .reset_index(drop=True)
)

dss_ranking.insert(0, "Rank", range(1, len(dss_ranking) + 1))

# Keep only the paper-facing DSS matrix.
paper_columns = [
    "Rank", "Dataset", "Alert/Event", "Context",
    "Threat Intelligence", "Exploitability", "Historical",
    "Ground Truth", "Scalability", "DSS", "DSS (%)",
]
display(dss_ranking[paper_columns])


,Rank,Dataset,Alert/Event,Context,Threat Intelligence,Exploitability,Historical,Ground Truth,Scalability,DSS,DSS (%)
0,1,MITS-Network,5.0,5.0,4.0,0.0,4.0,4.0,1.0,3.70,74.0
1,2,BigFlow-Parquet,5.0,1.0,2.0,0.0,4.0,5.0,2.0,2.80,56.0
2,3,BCCC-CIC-IDS-2017,3.0,3.0,2.0,0.0,4.0,4.0,2.0,2.70,54.0
3,4,CIC-IoT2023,2.0,0.0,5.0,0.0,3.0,5.0,2.0,2.30,46.0
4,5,BCCC-CSE-IDS2018,2.0,0.0,0.0,0.0,3.0,4.0,2.0,1.45,29.0
5,6,TON-IoT,2.0,0.0,0.0,0.0,2.0,4.0,2.0,1.30,26.0
6,7,UNSW-NB15,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.20,4.0
